## Contents

- [Querying Nominatim for Coordinates from Place Names](#querying-nominatim-for-coordinates-from-place-names)
- [Querying OpenWeatherMap for Weather from Coordinates](#querying-openweathermap-for-weather-from-coordinates)
- [Scraping Booking.com](#scraping-bookingcom)
- [Making the Final Visualisations](#making-the-final-visualisations)
- [Loading the CSV Into an S3 Bucket](#loading-the-csv-into-an-s3-bucket)

In [34]:
import requests, os, json, boto3

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from tqdm import tqdm
from time import sleep
from random import randint
from scrapy import Selector
from dotenv import load_dotenv
from plotly.subplots import make_subplots

## Querying Nominatim for Coordinates from Place Names

In [ ]:
load_dotenv()

def get_coords(place_name):
    url = "https://nominatim.openstreetmap.org/search"
    params = {
        "q": place_name,
        "format": "json",
        "limit": 1
    }
    headers = {
        "User-Agent": f"ExamAssignment/1.0 ({os.environ["NOMINATIM_EMAIL"]})"
    }
    
    try:
        response = requests.get(url, params=params, headers=headers)
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        return e
    
    if data:
        lat = data[0]["lat"]
        lon = data[0]["lon"]
        return {"lat" : float(lat), "lon" : float(lon)}
    else:
        print("No results found.")
        return None

destination_list = ["Mont Saint Michel", "St Malo", "Bayeux", "Le Havre", "Rouen", "Paris", "Amiens", "Lille", "Strasbourg", "Chateau du Haut Koenigsbourg", "Colmar", "Eguisheim",
                    "Besancon", "Dijon", "Annecy", "Grenoble", "Lyon", "Gorges du Verdon", "Bormes les Mimosas", "Cassis", "Marseille", "Aix en Provence", "Avignon", "Uzes", "Nimes",
                    "Aigues Mortes", "Saintes Maries de la mer", "Collioure", "Carcassonne", "Ariege", "Toulouse", "Montauban", "Biarritz", "Bayonne", "La Rochelle"]

coords_dict = dict()

for destination in destination_list:
    coords_dict[destination] = get_coords(destination)
    sleep(5)

coords_dict


{'Mont Saint Michel': {'lat': 48.6359541, 'lon': -1.51146},
 'St Malo': {'lat': 49.314695, 'lon': -96.9538228},
 'Bayeux': {'lat': 49.2764624, 'lon': -0.7024738},
 'Le Havre': {'lat': 49.4938975, 'lon': 0.1079732},
 'Rouen': {'lat': 49.4404591, 'lon': 1.0939658},
 'Paris': {'lat': 48.8534951, 'lon': 2.3483915},
 'Amiens': {'lat': 49.8941708, 'lon': 2.2956951},
 'Lille': {'lat': 50.6365654, 'lon': 3.0635282},
 'Strasbourg': {'lat': 48.584614, 'lon': 7.7507127},
 'Chateau du Haut Koenigsbourg': {'lat': 48.2494107, 'lon': 7.3443202},
 'Colmar': {'lat': 48.0777517, 'lon': 7.3579641},
 'Eguisheim': {'lat': 48.0447968, 'lon': 7.3079618},
 'Besancon': {'lat': 47.2380222, 'lon': 6.0243622},
 'Dijon': {'lat': 47.3215806, 'lon': 5.0414701},
 'Annecy': {'lat': 45.8992348, 'lon': 6.1288847},
 'Grenoble': {'lat': 45.1875602, 'lon': 5.7357819},
 'Lyon': {'lat': 45.7578137, 'lon': 4.8320114},
 'Gorges du Verdon': {'lat': 43.7496562, 'lon': 6.3285616},
 'Bormes les Mimosas': {'lat': 43.1506968, 'lon':

## Querying OpenWeatherMap for Weather from Coordinates

In [ ]:
load_dotenv()

def get_weather(coords_as_dict: dict):
    url = "https://api.openweathermap.org/data/3.0/onecall"
    params = {
        "lat" : coords_as_dict["lat"],
        "lon" : coords_as_dict["lon"],
        "APPID" : os.environ["OPENWEATHER_API_KEY"],
        "units" : "metric",
        "exclude" : "current,minutely,hourly,alerts"
    }
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        return e
    
    if data:
        return data["daily"]
    else:
        print("No results found.")
        return None

weather_dict = coords_dict.copy()

for destination, coords in tqdm(weather_dict.items()):
    weather_dict[destination]["daily_forecast"] = get_weather(coords)
    sleep(5)

weather_dict

100%|██████████| 35/35 [02:59<00:00,  5.13s/it]


{'Mont Saint Michel': {'lat': 48.6359541,
  'lon': -1.51146,
  'daily_forecast': [{'dt': 1760526000,
    'sunrise': 1760509557,
    'sunset': 1760548639,
    'moonrise': 1760484300,
    'moonset': 1760541300,
    'moon_phase': 0.81,
    'summary': 'The day will start with partly cloudy through the late morning hours, transitioning to clearing',
    'temp': {'day': 17.03,
     'min': 11.25,
     'max': 17.52,
     'night': 12.15,
     'eve': 15.7,
     'morn': 11.48},
    'feels_like': {'day': 16.43, 'night': 11.54, 'eve': 15.13, 'morn': 10.69},
    'pressure': 1026,
    'humidity': 63,
    'dew_point': 6.78,
    'wind_speed': 5.91,
    'wind_deg': 46,
    'wind_gust': 10.82,
    'weather': [{'id': 800,
      'main': 'Clear',
      'description': 'clear sky',
      'icon': '01d'}],
    'clouds': 0,
    'pop': 0,
    'uvi': 2.68},
   {'dt': 1760612400,
    'sunrise': 1760596048,
    'sunset': 1760634922,
    'moonrise': 1760575380,
    'moonset': 1760628780,
    'moon_phase': 0.84,
    '

In [ ]:
with open("dict_backup.json", "w") as f:
    json.dump(obj=weather_dict, fp=f)

## Making a First DataFrame with Weather Data for Each Destination

In [ ]:
destination_list = ["Mont Saint Michel", "St Malo", "Bayeux", "Le Havre", "Rouen", "Paris", "Amiens", "Lille", "Strasbourg", "Chateau du Haut Koenigsbourg", "Colmar", "Eguisheim",
                    "Besancon", "Dijon", "Annecy", "Grenoble", "Lyon", "Gorges du Verdon", "Bormes les Mimosas", "Cassis", "Marseille", "Aix en Provence", "Avignon", "Uzes", "Nimes",
                    "Aigues Mortes", "Saintes Maries de la mer", "Collioure", "Carcassonne", "Ariege", "Toulouse", "Montauban", "Biarritz", "Bayonne", "La Rochelle"]

weather_df_dict = {"place_id" : ["msm", "sml", "byx", "lhv", "ron", "prs", "ams", "lll", "sbg", "chk", "clm", "egh", "bsc", "djn", "anc", "grb", "lyn", "gdv",
                                 "blm", "css", "msl", "xep", "avg", "uzs", "nms", "agm", "smm", "clr", "crc", "arg", "tls", "mtb", "brz", "byn", "lrc"],
                   "placename" : destination_list,
                   "lon" : [],
                   "lat" : [],
                   "mean_max_temp_over_next_7_days" : [],
                   "mean_cloud_cover_over_next_7_days" : [],
                   "mean_POP_over_next_7_days" : [],
                   "bad_weather_score" : []}

for destination in destination_list:
    destination_data = weather_dict[destination]
    
    weather_df_dict["lon"].append(destination_data["lon"])
    weather_df_dict["lat"].append(destination_data["lat"])

    mean_max_temp_over_next_7_days = np.mean([day_forecast["temp"]["max"]
                                              for day_forecast in destination_data["daily_forecast"]])
    weather_df_dict["mean_max_temp_over_next_7_days"].append(mean_max_temp_over_next_7_days)

    mean_cloud_cover_over_next_7_days = np.mean([day_forecast["clouds"]
                                                 for day_forecast in destination_data["daily_forecast"]])
    weather_df_dict["mean_cloud_cover_over_next_7_days"].append(mean_cloud_cover_over_next_7_days)

    mean_POP_over_next_7_days = np.mean([day_forecast["temp"]["max"]
                                         for day_forecast in destination_data["daily_forecast"]])
    weather_df_dict["mean_POP_over_next_7_days"].append(mean_POP_over_next_7_days)

    # The instructions say that I can decide what "good weather" is, so I decided purely based on personal taste that an acceptable
    # temperature range is between 0ºC and 30ºC. If the temperature goes above 30, the score starts increasing again.
    # Since there are two "bad weather" indicators that correspond to percentages and one "good weather" indicator that doesn't,
    # I decided to modify the "good weather" value so it would be out of 100 and go down as it gets better, and thus match.
    weather_df_dict["bad_weather_score"].append(np.mean([abs(100 - ((mean_max_temp_over_next_7_days / 30) * 100)),
                                                         mean_cloud_cover_over_next_7_days,
                                                         mean_POP_over_next_7_days]))



weather_df = pd.DataFrame(weather_df_dict).sort_values(by="bad_weather_score", ascending=True)\
                                          .reset_index(drop=True)
weather_df

,place_id,placename,lon,lat,mean_max_temp_over_next_7_days,mean_cloud_cover_over_next_7_days,mean_POP_over_next_7_days,bad_weather_score
0,xep,Aix en Provence,5.447474,43.529842,20.47125,13.500,20.47125,21.911250
1,msl,Marseille,5.369953,43.296174,20.13375,15.375,20.13375,22.798750
2,avg,Avignon,4.805901,43.949249,21.85250,21.750,21.85250,23.586944
3,css,Cassis,5.539632,43.214036,19.26125,17.250,19.26125,24.102361
4,blm,Bormes les Mimosas,6.341928,43.150697,18.95125,16.750,18.95125,24.176806
5,nms,Nimes,4.360069,43.837425,22.26625,27.875,22.26625,25.306806
6,uzs,Uzes,4.419672,44.012128,21.23625,26.375,21.23625,25.607917
7,smm,Saintes Maries de la mer,4.427720,43.451592,19.32125,23.000,19.32125,25.972361
8,agm,Aigues Mortes,4.191540,43.566152,20.11000,28.125,20.11000,27.067222
9,tls,Toulouse,1.444243,43.604464,21.92250,38.000,21.92250,28.949167


In [ ]:
weather_df.to_csv("place_weather.csv")

In [90]:
weather_df = pd.read_csv("place_weather.csv")

fig = px.scatter_map(weather_df,
                     lon="POI_lon", 
                     lat="POI_lat",
                     color="bad_weather_score",
                     title="What You Can Expect From the Weather These Next 7 Days",
                     zoom=4.5,
                     height=700,
                     width=900,
                     size_max=10,
                     size=[10]*len(weather_df),
                     color_continuous_scale=px.colors.sequential.Blackbody,
                     opacity=1)
fig.write_html("7_day_weather.html")
fig.show()

## Scraping Booking.com

I initially scraped using an elegant set of URLs populated in a loop with the set of placenames, as shown below.\
I even got it to work in a rudimentary spider a few times, but that technique permanently started giving pages without results and asking me to prove I'm not a robot before I could finish the spider (even when rotating proxies and user agents, even late at night, even when waiting 24hrs, even when putting a delay with time.sleep(), etc.).\
I found a different results page with a different structure, but it never worked with spiders: I ended up scraping the website with a script that used GET requests and Scrapy Selectors, because that worked just fine and was easier to output data in a DataFrame-friendly format with.\
If I had more time, there would probably have been ways to use spiders, such as by interacting with the Javascript through scrapy-playwright or scrapy-splash, and by checking whether the scrapy project is actually doing the obfuscation things it has to.

In [76]:
for placename in weather_df.placename.to_list():
    print(requests.get(f"https://www.booking.com/searchresults.fr.html?ss={placename}"))
    sleep(3)

<Response [200]>
<Response [200]>


KeyboardInterrupt: 

In [ ]:
url = "https://www.booking.com/searchresults.fr.html?ss=%27Mont%20Saint%20Michel%27"
response = Selector(text=requests.get(url).text)

for item in response.xpath("//*[contains(@class, 'cca574b93c')]").getall(): # This class matches a pair of tags that contain all the results
    if "data-results-container" in item:
        print("found it!")

found it!


In [1]:
!python booking_spider.py

2025-10-16 00:38:12 [scrapy.utils.log] INFO: Scrapy 2.12.0 started (bot: scrapybot)
2025-10-16 00:38:12 [scrapy.utils.log] INFO: Versions: lxml 5.3.0.0, libxml2 2.13.8, cssselect 1.2.0, parsel 1.8.1, w3lib 2.1.2, Twisted 24.11.0, Python 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 11:09:21) [Clang 14.0.6 ], pyOpenSSL 25.1.0 (OpenSSL 3.5.4 30 Sep 2025), cryptography 45.0.5, Platform macOS-10.16-x86_64-i386-64bit-Mach-O
2025-10-16 00:38:12 [scrapy.addons] INFO: Enabled addons:
[]
2025-10-16 00:38:12 [scrapy.extensions.telnet] INFO: Telnet Password: 832683c52d26ed6b
2025-10-16 00:38:13 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
 'scrapy.extensions.telnet.TelnetConsole',
 'scrapy.extensions.memusage.MemoryUsage',
 'scrapy.extensions.feedexport.FeedExporter',
 'scrapy.extensions.logstats.LogStats']
2025-10-16 00:38:13 [scrapy.crawler] INFO: Overridden settings:
{'LOG_LEVEL': 20, 'USER_AGENT': 'Chrome/97.0'}
2025-10-16 00:38:13 [scrap

In [15]:
with open("./src/booking_locations.json", "r") as f:
    results_dict = json.load(f)

In [ ]:
html_str = results_dict[0]["dummy_response"]
response = Selector(text=html_str)
responses = response.xpath("//*[contains(@class, 'c3bdfd4ac2 a0ab5da06c d46ff48a92 f728e61e72 d0acd69e66 c256f1a28a bc2204a477')]/div[1]/div[2]/div[1]/div[1]/div[1]/div/div[3]/text()")
for r in responses:
    print(r.get())

Installé dans des espaces verts à seulement 2 km de l'abbaye du Mont-Saint-Michel, le Mercure Mont Saint Michel propose des chambres spacieuses avec salle de bains privative, un bar-restaurant et de...
Situé au Mont-Saint-Michel, l’établissement 2 étoiles Le Duguesclin possède un restaurant et un bar. 
La Vieille Auberge vous accueille dans le village médiéval du Mont-Saint-Michel, à quelques pas de la célèbre abbaye. 
Séjournez dans un hôtel historique au cœur du Mont Saint-Michel. L'Hôtel La Mère Poulard, situé au cœur du Mont Saint-Michel, a conservé le charme authentique des auberges traditionnelles d'autrefois....
L’Hotel Vert vous propose des chambres décorées dans des tons pastel, dotées d’une salle de bains privative, d’une télévision ainsi que d’une connexion Wi-Fi gratuite. 
Occupant 2 bâtiments différents au cœur du Mont-Saint-Michel, l’établissement historique Les Terrasses Poulard propose des hébergements avec une vue sur la baie, le village et la rue. 
Le Relais Du Roy es

#### Trying to Access the Button to Load More Results

It turns out that it only appears if the user scrolls down, but scrapy alone doesn't allow one to interact with the website other than reading a frozen version of the HTML.

In [ ]:
url = "https://www.booking.com/searchresults.fr.html?ss=%27Mont%20Saint%20Michel%27"

response = requests.get(url).text

("Afficher plus de résultats" in response)

False

In [55]:
response.find("Afficher plus de résultats")

550829

In [57]:
response[550800:550900]

'"sr_mdot_load_more_results":"Afficher plus de résultats","sr_nodates_show_prices":"Voir les tarifs",'

#### Researching How to Access the Coordinates of Each Hotel, Which Requires Loading Each Hotel Page

In [7]:
url = "https://www.booking.com/hotel/fr/le-marquis-de-la-guintre.fr.html?aid=304142&label=gen173nr-10CAQoggJCHXNlYXJjaF8nbGUgbW9udCBzYWludCBtaWNoZWwnSA1YBGhNiAEBmAEzuAEKyAEF2AED6AEB-AEBiAIBqAIBuAK-xsDHBsACAdICJDA4MjZmMTY5LTgzZjItNDBiMy1iYTkwLTI0ZTU5NDIwOTUwNtgCAeACAQ&ucfs=1&arphpl=1&group_adults=2&req_adults=2&no_rooms=1&group_children=0&req_children=0&hpos=20&hapos=20&sr_order=popularity&srpvid=1f089f9f97140053&srepoch=1760568128&from=searchresults"
coords_response = Selector(text=requests.get(url).text)

In [8]:
coords_responses = coords_response.xpath("//*[contains(@id, 'map_trigger_header')]").attrib["data-atlas-latlng"]
print(coords_responses.split(","))

['48.6248647057706', '-1.44534185528755']


In [ ]:
# This is a separate spider file that I was planning on calling from within the main spider,
# in order to scrape the coordinates and return them to the main spider
import hotel_coordinates_spider

result = hotel_coordinates_spider.get("https://www.booking.com/hotel/fr/le-marquis-de-la-guintre.fr.html?aid=304142&label=gen173nr-10CAQoggJCHXNlYXJjaF8nbGUgbW9udCBzYWludCBtaWNoZWwnSA1YBGhNiAEBmAEzuAEKyAEF2AED6AEB-AEBiAIBqAIBuAK-xsDHBsACAdICJDA4MjZmMTY5LTgzZjItNDBiMy1iYTkwLTI0ZTU5NDIwOTUwNtgCAeACAQ&ucfs=1&arphpl=1&group_adults=2&req_adults=2&no_rooms=1&group_children=0&req_children=0&hpos=20&hapos=20&sr_order=popularity&srpvid=1f089f9f97140053&srepoch=1760568128&from=searchresults")

In [2]:
("map_trigger_header" in result)

False

#### Since No Spider is Working, Even When Rotating Proxies and User Agents, but GET Requests Paired With Scrapy Selectors Work Fine, I'll Scrape Using the Latter and Try Again Later

In [ ]:
url = "https://www.booking.com/searchresults.fr.html?ss=Bayeux&ssne=Bayeux&ssne_untouched=Bayeux&highlighted_hotels=9807677&label=gen000nr-10CAsoTUINbGUtcGV0aXQtbWFsb0gNWARoTYgBAZgBM7gBBcgBHtgBA-gBAfgBAYgCAagCAbgC3Z_IxwbAAgHSAiQzNWNhYmJmNy05MmU2LTQ2NTQtOGViZi0wNDk0OWYwYjRjMTbYAgHgAgE&aid=304142&lang=fr&sb=1&src_elem=sb&src=hotel&dest_id=-1410836&dest_type=city&group_adults=2&no_rooms=1&group_children=0"
coords_response = Selector(text=requests.get(url).text)

coords_responses = coords_response.xpath("//*[contains(@class, 'b071d3eccd afd3558156 ecc7830bbb')]/div[1]/div[1]/div[1]/div[1]/a/div[1]/h3/div[1]/text()").getall()
print(coords_responses)

['Domaine de Bayeux', 'Villa Lara Hotel', 'Hôtel De Brunville & Spa', "Hotel Le Lion D'Or et Restaurant La Table Du Lion", 'Hotel Reine Mathilde', "Hôtel d'Argouges", 'Belle Normandy', 'Le Declic', 'La Maison de Mathilde', 'Hôtel Le Saint Patrice']


In [53]:
len(coords_responses)

10

#### What Did the Job

In [ ]:
# Programmatically getting or making search page URLs didn't work, so I resorted to manually getting them.
# If I had hundreds or thousands of those to get, I would have used Selenium, scrapy-playwright or scrapy-splash
# to programmatically write the place names in the search field, press the search button, and scrape what came up.

# I of course tried simply changing the placenames, like I was initially doing, but that doesn't give hotel results.

urls_dict = {
    "Mont Saint Michel" : "https://www.booking.com/searchresults.fr.html?ss=Le+Mont-Saint-Michel%2C+Basse-Normandie%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=index&dest_id=900039327&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=7ed95886238f0057&ac_meta=GhA3ZWQ5NTg4NjIzOGYwMDU3IAAoATICZnI6B2xlIG1vbnRAAEoAUAA%3D&group_adults=2&no_rooms=1&group_children=0",
    "St Malo" : "https://www.booking.com/searchresults.fr.html?ss=Saint-Malo%2C+Bretagne%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1466824&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=b3c958dde02a00f2&ac_meta=GhBiM2M5NThkZGUwMmEwMGYyIAAoATICZnI6BXNhaW50QABKAFAA&group_adults=2&no_rooms=1&group_children=0",
    "Bayeux" : "https://www.booking.com/searchresults.fr.html?ss=Bayeux%2C+Basse-Normandie%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1410836&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=67c458f868cc0234&ac_meta=GhA2N2M0NThmODY4Y2MwMjM0IAAoATICZnI6BmJheWV1eEAASgBQAA%3D%3D&group_adults=2&no_rooms=1&group_children=0",
    "Le Havre" : "https://www.booking.com/searchresults.fr.html?ss=Le+Havre%2C+Haute-Normandie%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1441598&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=551f59c5b0ce1b9c&ac_meta=GhA1NTFmNTljNWIwY2UxYjljIAAoATICZnI6CExlIGhhdnJlQABKAFAA&group_adults=2&no_rooms=1&group_children=0",
    "Rouen" : "https://www.booking.com/searchresults.fr.html?ss=Rouen%2C+Haute-Normandie%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1462807&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=ce3f59f3d9230239&ac_meta=GhBjZTNmNTlmM2Q5MjMwMjM5IAAoATICZnI6BXJvdWVuQABKAFAA&group_adults=2&no_rooms=1&group_children=0",
    "Paris" : "https://www.booking.com/searchresults.fr.html?ss=Paris&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1456928&dest_type=city&group_adults=2&no_rooms=1&group_children=0",
    "Amiens" : "https://www.booking.com/searchresults.fr.html?ss=Amiens%2C+Picardie%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1407447&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=8df858d3a0f304c6&ac_meta=GhA4ZGY4NThkM2EwZjMwNGM2IAAoATICZnI6BmFtaWVuc0AASgBQAA%3D%3D&group_adults=2&no_rooms=1&group_children=0",
    "Lille" : "https://www.booking.com/searchresults.fr.html?ss=Lille%2C+Nord-Pas-de-Calais%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1447079&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=c84b5b8161dd010a&ac_meta=GhBjODRiNWI4MTYxZGQwMTBhIAAoATICZnI6BUxpbGxlQABKAFAA&group_adults=2&no_rooms=1&group_children=0",
    "Strasbourg" : "https://www.booking.com/searchresults.fr.html?ss=Strasbourg%2C+Alsace%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1471697&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=17e25b86c68f0268&ac_meta=GhAxN2UyNWI4NmM2OGYwMjY4IAAoATICZnI6ClN0cmFzYm91cmdAAEoAUAA%3D&group_adults=2&no_rooms=1&group_children=0",
    "Chateau du Haut Koenigsbourg" : "https://www.booking.com/searchresults.fr.html?ss=Ch%C3%A2teau+du+Haut-K%C5%93nigsbourg%2C+Saint-Hippolyte%2C+Alsace%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=204055&dest_type=landmark&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=4&search_selected=true&search_pageview_id=c8265bac979f04ca&ac_meta=GhBjODI2NWJhYzk3OWYwNGNhIAAoATICZnI6HENoYXRlYXUgZHUgSGF1dCBLb2VuaWdzYm91cmdAAEoAUAA%3D&group_adults=2&no_rooms=1&group_children=0",
    "Colmar" : "https://www.booking.com/searchresults.fr.html?ss=Colmar%2C+Alsace%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1421049&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=e9e05bcb952204e0&ac_meta=GhBlOWUwNWJjYjk1MjIwNGUwIAAoATICZnI6BkNvbG1hckAASgBQAA%3D%3D&group_adults=2&no_rooms=1&group_children=0",
    "Eguisheim" : "https://www.booking.com/searchresults.fr.html?ss=Eguisheim%2C+Alsace%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1425030&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=5f365bf98ca301c7&ac_meta=GhA1ZjM2NWJmOThjYTMwMWM3IAAoATICZnI6B2VndWlzaGVAAEoAUAA%3D&group_adults=2&no_rooms=1&group_children=0",
    "Besancon" : "https://www.booking.com/searchresults.fr.html?ss=Besan%C3%A7on%2C+Franche-Comt%C3%A9%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1412198&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=accc5c2e1a2a0181&ac_meta=GhBhY2NjNWMyZTFhMmEwMTgxIAAoATICZnI6BUJlc2FuQABKAFAA&group_adults=2&no_rooms=1&group_children=0",
    "Dijon" : "https://www.booking.com/searchresults.fr.html?ss=Dijon%2C+Bourgogne%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1423981&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=3a595c6d9f1a0373&ac_meta=GhAzYTU5NWM2ZDlmMWEwMzczIAAoATICZnI6BURpam9uQABKAFAA&group_adults=2&no_rooms=1&group_children=0",
    "Annecy" : "https://www.booking.com/searchresults.fr.html?ss=Annecy%2C+Rh%C3%B4ne-Alpes%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1407760&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=ed215c8a91ec0412&ac_meta=GhBlZDIxNWM4YTkxZWMwNDEyIAAoATICZnI6BkFubmVjeUAASgBQAA%3D%3D&group_adults=2&no_rooms=1&group_children=0",
    "Grenoble" : "https://www.booking.com/searchresults.fr.html?ss=Grenoble%2C+Rh%C3%B4ne-Alpes%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1430647&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=a6f15c909e380171&ac_meta=GhBhNmYxNWM5MDllMzgwMTcxIAAoATICZnI6CEdyZW5vYmxlQABKAFAA&group_adults=2&no_rooms=1&group_children=0",
    "Lyon" : "https://www.booking.com/searchresults.fr.html?ss=Lyon%2C+Rh%C3%B4ne-Alpes%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1448468&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=9fc25ce0325519a5&ac_meta=GhA5ZmMyNWNlMDMyNTUxOWE1IAAoATICZnI6BEx5b25AAEoAUAA%3D&group_adults=2&no_rooms=1&group_children=0",
    "Gorges du Verdon" : "https://www.booking.com/searchresults.fr.html?ss=Gorges+du+Verdon%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=2746&dest_type=region&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=ca845d59d3e0082a&ac_meta=GhBjYTg0NWQ1OWQzZTAwODJhIAAoATICZnI6CmdvcmdlcyBkdSBAAEoAUAA%3D&group_adults=2&no_rooms=1&group_children=0",
    "Bormes les Mimosas" : "https://www.booking.com/searchresults.fr.html?ss=Bormes-les-Mimosas%2C+Provence-Alpes-C%C3%B4te+d%27Azur%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1413801&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=d2b95d8c681004ba&ac_meta=GhBkMmI5NWQ4YzY4MTAwNGJhIAAoATICZnI6DmJvcm1lcyBsZXMgbWltQABKAFAA&group_adults=2&no_rooms=1&group_children=0",
    "Cassis" : "https://www.booking.com/searchresults.fr.html?ss=Cassis%2C+Provence-Alpes-C%C3%B4te+d%27Azur%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1416912&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=d4b25e043f2e0578&ac_meta=GhBkNGIyNWUwNDNmMmUwNTc4IAAoATICZnI6BkNhc3Npc0AASgBQAA%3D%3D&group_adults=2&no_rooms=1&group_children=0",
    "Marseille" : "https://www.booking.com/searchresults.fr.html?ss=Marseille%2C+Provence-Alpes-C%C3%B4te+d%27Azur%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1449947&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=4f705e1f2592000e&ac_meta=GhA0ZjcwNWUxZjI1OTIwMDBlIAAoATICZnI6CW1hcnNlaWxsZUAASgBQAA%3D%3D&group_adults=2&no_rooms=1&group_children=0",
    "Aix en Provence" : "https://www.booking.com/searchresults.fr.html?ss=Aix-en-Provence%2C+Provence-Alpes-C%C3%B4te+d%27Azur%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1406939&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=2c455e36447f0190&ac_meta=GhAyYzQ1NWUzNjQ0N2YwMTkwIAAoATICZnI6BmFpeCBlbkAASgBQAA%3D%3D&group_adults=2&no_rooms=1&group_children=0",
    "Avignon" : "https://www.booking.com/searchresults.fr.html?ss=Avignon%2C+Provence-Alpes-C%C3%B4te+d%27Azur%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1409631&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=ab305e5809190326&ac_meta=GhBhYjMwNWU1ODA5MTkwMzI2IAAoATICZnI6B0F2aWdub25AAEoAUAA%3D&group_adults=2&no_rooms=1&group_children=0",
    "Uzes" : "https://www.booking.com/searchresults.fr.html?ss=Uz%C3%A8s%2C+Languedoc-Roussillon%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1474231&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=427b5e5cd0cb0443&ac_meta=GhA0MjdiNWU1Y2QwY2IwNDQzIAAoATICZnI6BHV6ZXNAAEoAUAA%3D&group_adults=2&no_rooms=1&group_children=0",
    "Nimes" : "https://www.booking.com/searchresults.fr.html?ss=N%C3%AEmes%2C+Languedoc-Roussillon%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1455068&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=ab305e6c190d023a&ac_meta=GhBhYjMwNWU2YzE5MGQwMjNhIAAoATICZnI6BW5pbWVzQABKAFAA&group_adults=2&no_rooms=1&group_children=0",
    "Aigues Mortes" : "https://www.booking.com/searchresults.fr.html?ss=Aigues-Mortes%2C+Languedoc-Roussillon%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1406800&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=7c9c5e84e36d0537&ac_meta=GhA3YzljNWU4NGUzNmQwNTM3IAAoATICZnI6BmFpZ3Vlc0AASgBQAA%3D%3D&group_adults=2&no_rooms=1&group_children=0",
    "Saintes Maries de la mer" : "https://www.booking.com/searchresults.fr.html?ss=Les+Saintes-Maries-de-la-Mer%2C+Provence-Alpes-C%C3%B4te+d%27Azur%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1465138&dest_type=city&ac_position=1&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=190c5e9734b11046&ac_meta=GhAxOTBjNWU5NzM0YjExMDQ2IAEoATICZnI6B3NhaW50ZXNAAEoAUAA%3D&group_adults=2&no_rooms=1&group_children=0",
    "Collioure" : "https://www.booking.com/searchresults.fr.html?ss=Collioure%2C+Languedoc-Roussillon%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1421032&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=c8265ea8a19a000b&ac_meta=GhBjODI2NWVhOGExOWEwMDBiIAAoATICZnI6CUNvbGxpb3VyZUAASgBQAA%3D%3D&group_adults=2&no_rooms=1&group_children=0",
    "Carcassonne" : "https://www.booking.com/searchresults.fr.html?ss=Carcassonne%2C+Languedoc-Roussillon%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1416701&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=54b45ebe8687040b&ac_meta=GhA1NGI0NWViZTg2ODcwNDBiIAAoATICZnI6C2NhcmNhc3Nvbm5lQABKAFAA&group_adults=2&no_rooms=1&group_children=0",
    "Ariege" : "https://www.booking.com/searchresults.fr.html?ss=Ari%C3%A8ge%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=2507&dest_type=region&ac_position=2&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=e19b5ef0f9ba04ce&ac_meta=GhBlMTliNWVmMGY5YmEwNGNlIAIoATICZnI6BUFyacOoQABKAFAA&group_adults=2&no_rooms=1&group_children=0",
    "Toulouse" : "https://www.booking.com/searchresults.fr.html?ss=Toulouse%2C+Midi-Pyr%C3%A9n%C3%A9es%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1473166&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=80465f4df81f00d5&ac_meta=GhA4MDQ2NWY0ZGY4MWYwMGQ1IAAoATICZnI6CFRvdWxvdXNlQABKAFAA&group_adults=2&no_rooms=1&group_children=0",
    "Montauban" : "https://www.booking.com/searchresults.fr.html?ss=Montauban%2C+Midi-Pyr%C3%A9n%C3%A9es%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1452421&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=c62b5f587d860468&ac_meta=GhBjNjJiNWY1ODdkODYwNDY4IAAoATICZnI6CW1vbnRhdWJhbkAASgBQAA%3D%3D&group_adults=2&no_rooms=1&group_children=0",
    "Biarritz" : "https://www.booking.com/searchresults.fr.html?ss=Biarritz%2C+Aquitaine%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1412526&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=67c45f69d27202ce&ac_meta=GhA2N2M0NWY2OWQyNzIwMmNlIAAoATICZnI6BmJpYXJyaUAASgBQAA%3D%3D&group_adults=2&no_rooms=1&group_children=0",
    "Bayonne" : "https://www.booking.com/searchresults.fr.html?ss=Bayonne%2C+Aquitaine%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1410844&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=bc625f78768c04c3&ac_meta=GhBiYzYyNWY3ODc2OGMwNGMzIAAoATICZnI6B2JheW9ubmVAAEoAUAA%3D&group_adults=2&no_rooms=1&group_children=0",
    "La Rochelle" : "https://www.booking.com/searchresults.fr.html?ss=La+Rochelle%2C+Poitou-Charentes%2C+France&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaE2IAQGYATO4AQfIAQ_YAQPoAQH4AQGIAgGoAgG4AozwyMcGwAIB0gIkN2ZlNDhmMDQtNTg3Mi00NzI3LWExNWEtZjI4YjNiNDc4YTRh2AIB4AIB&sid=ec92b03eccf337e979d9b3cac5b2d333&aid=304142&lang=fr&sb=1&src_elem=sb&src=searchresults&dest_id=-1438604&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=fr&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=6fad5f8eb1120e41&ac_meta=GhA2ZmFkNWY4ZWIxMTIwZTQxIAAoATICZnI6C2xhIHJvY2hlbGxlQABKAFAA&group_adults=2&no_rooms=1&group_children=0"
}

In [ ]:
# This did a much better job at scraping Booking.com, ironically

hotels_df_dict = {"placename" : [],
                  "hotel_name" : [],
                  "hotel_bookingdotcom_page" : [],
                  "user_rating" : [],
                  "hotel_lat" : [],
                  "hotel_lon" : [],
                  "description" : []}

for placename, url in urls_dict.items():
    response = Selector(text=requests.get(url).text)
    hotels = response.xpath("//*[contains(@class, 'b071d3eccd afd3558156 ecc7830bbb')]/div[1]/div[1]/div[1]")

    print(f"Got hotels HTML for {placename}\n{hotels.get() is None}")

    for hotel in hotels:
        hotel_url = hotel.xpath("div[1]/a").attrib["href"]
        hotels_df_dict["placename"].append(placename)
        hotels_df_dict["hotel_name"].append(hotel.xpath("div[1]/a/div[1]/h3/div[1]/text()").get())
        hotels_df_dict["hotel_bookingdotcom_page"].append(hotel_url)
        hotels_df_dict["user_rating"].append(float(hotel.xpath("div[2]/div[1]/div[1]/div[2]/text()").get().replace(",", ".")))
        hotels_df_dict["hotel_description"].append(hotel.xpath("div[1]/div[1]/p/text()").get())
        print("Added data to df!")
        sleep(randint(1, 5))
        coordinates_response = Selector(text=requests.get(hotel_url).text)
        try:
            latlon = coordinates_response.xpath("//*[contains(@id, 'map_trigger_header')]")\
                                        .attrib["data-atlas-latlng"]\
                                        .split(",")
        except:
            print(coordinates_response.xpath("/html").get())
        hotels_df_dict["hotel_lat"].append(float(latlon[0]))
        hotels_df_dict["hotel_lon"].append(float(latlon[1]))
        print(latlon)

    sleep(randint(5, 12))

hotels_df = pd.DataFrame(hotels_df_dict)
hotels_df.to_csv("place_hotel2.csv")

Got hotels HTML for Mont Saint Michel
False
Added data to df!
['48.61470048629041', '-1.5096169710159302']
Added data to df!
['48.6175872716489', '-1.51039615273476']
Added data to df!
['48.61538141368341', '-1.510709971189499']
Added data to df!
['48.61424652959294', '-1.510545015335083']
Added data to df!
['48.61626270451236', '-1.510905772447586']
Added data to df!
['48.612937834706464', '-1.5101051330566406']
Added data to df!
['48.63534942564122', '-1.5103787183761597']
Added data to df!
['48.63508531723403', '-1.5105396509170532']
Added data to df!
['48.636022984447116', '-1.509895920753479']
Added data to df!
['48.6354874', '-1.5101555']
Got hotels HTML for St Malo
False
Added data to df!
['48.63112529789551', '-2.010825276374817']
Added data to df!
['48.64961099853388', '-2.024303376674652']
Added data to df!
['48.650727056010815', '-2.0249454730163734']
Added data to df!
['48.638587743159896', '-1.96755051612854']
Added data to df!
['48.6495690720354', '-2.02686496197134']
Add

In [85]:
hotels_df.head(20)

,placename,hotel_name,hotel_bookingdotcom_page,user_rating,hotel_lat,hotel_lon,description
0,Mont Saint Michel,Hôtel Vert,https://www.booking.com/hotel/fr/vert.fr.html,"8,1",48.614700,-1.509617,L’Hotel Vert vous propose des chambres décorée...
1,Mont Saint Michel,Le Relais Saint Michel,https://www.booking.com/hotel/fr/le-relais-sai...,"8,2",48.617587,-1.510396,Le Relais Saint Michel vous accueille face à l...
2,Mont Saint Michel,Hotel Gabriel,https://www.booking.com/hotel/fr/hotel-gabriel...,"8,3",48.615381,-1.510710,Hotel Gabriel is located 1.6 Km from Mont Sain...
3,Mont Saint Michel,Mercure Mont Saint Michel,https://www.booking.com/hotel/fr/mont-saint-mi...,"8,3",48.614247,-1.510545,Installé dans des espaces verts à seulement 2 ...
4,Mont Saint Michel,Le Relais Du Roy,https://www.booking.com/hotel/fr/le-relais-du-...,"8,3",48.616263,-1.510906,Le Relais Du Roy est un hôtel 3 étoiles situé ...
5,Mont Saint Michel,Le Saint Aubert,https://www.booking.com/hotel/fr/hotel-saint-a...,"7,6",48.612938,-1.510105,"Niché dans un écrin de verdure, à seulement 2 ..."
6,Mont Saint Michel,Les Terrasses Poulard,https://www.booking.com/hotel/fr/les-terrasses...,"7,3",48.635349,-1.510379,Occupant 2 bâtiments différents au cœur du Mon...
7,Mont Saint Michel,La Mère Poulard,https://www.booking.com/hotel/fr/la-mere-poula...,"7,6",48.635085,-1.510540,Séjournez dans un hôtel historique au cœur du ...
8,Mont Saint Michel,Le Mouton Blanc,https://www.booking.com/hotel/fr/le-mouton-bla...,"7,2",48.636023,-1.509896,"Situé au pied de l’abbaye, l’hôtel Le Mouton B..."
9,Mont Saint Michel,Appart Standing - La Coque d'Or,https://www.booking.com/hotel/fr/la-coque-d-or...,"9,6",48.635487,-1.510155,L’hébergement Appart Standing - La Coque d'Or ...


In [ ]:
full_df = pd.merge(left=pd.read_csv("place_hotel.csv", index_col="Unnamed: 0"),
                   right=pd.read_csv("place_weather.csv", index_col="Unnamed: 0"),
                   how="left",
                   on="placename")
full_df.head()

,Unnamed: 0,placename,hotel_name,hotel_bookingdotcom_page,user_rating,hotel_lat,hotel_lon,hotel_description,place_id,POI_lon,POI_lat,mean_max_temp_over_next_7_days,mean_cloud_cover_over_next_7_days,mean_POP_over_next_7_days,bad_weather_score
0,0,Mont Saint Michel,Hôtel Vert,https://www.booking.com/hotel/fr/vert.fr.html,8.1,48.614700,-1.509617,L’Hotel Vert vous propose des chambres décorée...,msm,-1.51146,48.635954,18.3325,51.875,18.3325,36.366389
1,1,Mont Saint Michel,Le Relais Saint Michel,https://www.booking.com/hotel/fr/le-relais-sai...,8.2,48.617587,-1.510396,Le Relais Saint Michel vous accueille face à l...,msm,-1.51146,48.635954,18.3325,51.875,18.3325,36.366389
2,2,Mont Saint Michel,Hotel Gabriel,https://www.booking.com/hotel/fr/hotel-gabriel...,8.3,48.615381,-1.510710,Hotel Gabriel is located 1.6 Km from Mont Sain...,msm,-1.51146,48.635954,18.3325,51.875,18.3325,36.366389
3,3,Mont Saint Michel,Mercure Mont Saint Michel,https://www.booking.com/hotel/fr/mont-saint-mi...,8.3,48.614247,-1.510545,Installé dans des espaces verts à seulement 2 ...,msm,-1.51146,48.635954,18.3325,51.875,18.3325,36.366389
4,4,Mont Saint Michel,Le Relais Du Roy,https://www.booking.com/hotel/fr/le-relais-du-...,8.3,48.616263,-1.510906,Le Relais Du Roy est un hôtel 3 étoiles situé ...,msm,-1.51146,48.635954,18.3325,51.875,18.3325,36.366389


## Loading the CSV Into an S3 Bucket

In [17]:
load_dotenv()

session = boto3.Session(aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
                        aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"])
s3 = session.resource("s3")
bucket = s3.create_bucket(Bucket="jedha-kayak-cert",
                          CreateBucketConfiguration={"LocationConstraint": "eu-west-3"}) 
bucket.upload_file("full_data.csv", "full_data.csv")

## Making the Final Visualisations

In [ ]:
df = pd.read_csv("full_data.csv")

In [4]:
df.head()

,Unnamed: 0,placename,hotel_name,hotel_bookingdotcom_page,user_rating,hotel_lat,hotel_lon,hotel_description,place_id,POI_lon,POI_lat,mean_max_temp_over_next_7_days,mean_cloud_cover_over_next_7_days,mean_POP_over_next_7_days,bad_weather_score
0,0,Mont Saint Michel,Hôtel Vert,https://www.booking.com/hotel/fr/vert.fr.html,8.1,48.614700,-1.509617,L’Hotel Vert vous propose des chambres décorée...,msm,-1.51146,48.635954,18.3325,51.875,18.3325,36.366389
1,1,Mont Saint Michel,Le Relais Saint Michel,https://www.booking.com/hotel/fr/le-relais-sai...,8.2,48.617587,-1.510396,Le Relais Saint Michel vous accueille face à l...,msm,-1.51146,48.635954,18.3325,51.875,18.3325,36.366389
2,2,Mont Saint Michel,Hotel Gabriel,https://www.booking.com/hotel/fr/hotel-gabriel...,8.3,48.615381,-1.510710,Hotel Gabriel is located 1.6 Km from Mont Sain...,msm,-1.51146,48.635954,18.3325,51.875,18.3325,36.366389
3,3,Mont Saint Michel,Mercure Mont Saint Michel,https://www.booking.com/hotel/fr/mont-saint-mi...,8.3,48.614247,-1.510545,Installé dans des espaces verts à seulement 2 ...,msm,-1.51146,48.635954,18.3325,51.875,18.3325,36.366389
4,4,Mont Saint Michel,Le Relais Du Roy,https://www.booking.com/hotel/fr/le-relais-du-...,8.3,48.616263,-1.510906,Le Relais Du Roy est un hôtel 3 étoiles situé ...,msm,-1.51146,48.635954,18.3325,51.875,18.3325,36.366389


In [ ]:
top_5_destinations_df = df.groupby("placename", as_index=False)[["bad_weather_score", "POI_lon", "POI_lat"]]\
                       .mean()\
                       .sort_values(by="bad_weather_score", ascending=True)\
                       .drop(columns="bad_weather_score")\
                       .head()\
                       .reset_index(drop=True)

top_5_destinations_df

,placename,POI_lon,POI_lat
0,Aix en Provence,5.447474,43.529842
1,Marseille,5.369953,43.296174
2,Avignon,4.805901,43.949249
3,Cassis,5.539632,43.214036
4,Bormes les Mimosas,6.341928,43.150697


In [82]:
fig = px.scatter_map(data_frame=top_5_destinations_df,
                     lon="POI_lon",
                     lat="POI_lat",
                     zoom=7,
                     width=800,
                     height=600,
                     size_max=15,
                     size=[15]*len(top_5_destinations_df),
                     title="The Top 5 Destinations in the Data in Terms of Weather",
                     hover_name="placename")
fig.write_html("top_5_destinations.html")
fig.show()

In [ ]:
top_20_hotels_df = (df[df.placename.isin(top_5_destinations_df.placename.to_list())].sort_values(by="user_rating", ascending=False)\
                                                                        .head(20))

num_rows, num_cols = 3, 2

specs = [[{"type": "scattermap"} for _ in range(num_cols)]
         for _ in range(num_rows)]

fig = make_subplots(rows=num_rows,
                    cols=num_cols,
                    specs=specs)

for index, placename in enumerate(top_5_destinations_df.placename.to_list()):
    subset_df = top_20_hotels_df[top_20_hotels_df.placename == placename]
    
    lon = subset_df.hotel_lon
    lat = subset_df.hotel_lat
    
    fig.add_trace(go.Scattermap(lon=lon,
                                lat=lat,
                                mode="markers",
                                marker=dict(size=10, opacity=0.5),
                                text=subset_df.hotel_name,
                                name=placename
                                ), row=(index // num_cols) + 1,
                                   col=(index % num_cols) + 1)
    map_name = f"map{index + 1}"
    fig.update_layout({map_name : dict(center={"lon" : np.mean(lon),
                                              "lat" : np.mean(lat)},
                                      zoom=10)})

fig.update_layout(height=1000,
                  width=1500,
                  title_text="The Top 20 Hotels in the Data in Terms of Weather and User Ratings")
fig.write_html("top_20_hotels.html")
fig.show()
